# Start of Code

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
from pyspark.sql import SparkSession
import requests
import json

# Create a Spark session
spark = SparkSession.builder \
    .appName("read local json") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.driver.memory", "4g") \
    .master("local[*]") \
    .getOrCreate()

spark

spark.read.json("/kaggle/input/yelp-dataset/yelp_academic_dataset_review.json").show(10, False)

In [ ]:
academic_dataset_review = spark.read.json("/kaggle/input/yelp-dataset/yelp_academic_dataset_review.json")
recCount = academic_dataset_review.count()
print("Total number of records in the dataset: ", recCount)

In [ ]:
from pyspark.sql.functions import monotonically_increasing_id
import pandas as pd

# Add an index column to uniquely identify rows
df_with_index = academic_dataset_review.withColumn("index", monotonically_increasing_id())

# Define batch size
batch_size = 10000
total_records = academic_dataset_review.count()

for start in range(0, total_records, batch_size):
    # Filter the DataFrame for the current batch
    batch = df_with_index.filter(
        (df_with_index["index"] >= start) & (df_with_index["index"] < start + batch_size)
    )
    
    # Convert current batch to Pandas DataFrame
    pandas_batch_academic_dataset_review = batch.drop("index").toPandas()  # Drop index column
    
    # Perform operations on the Pandas DataFrame
    print(f"Processing batch: {start // batch_size}")
    print(pandas_batch_academic_dataset_review.head())  # Example: Display first few rows

In [ ]:
import sqlite3

# Create a SQLite DB in Kaggle's working directory
conn = sqlite3.connect("yelp.db")
cursor = conn.cursor()
# Save to SQLite database
table_name = f"tbl_academic_dataset_review"
pandas_batch_academic_dataset_review.to_sql(table_name, conn, if_exists='replace', index=False)
print(f"Table academic_dataset_review has been written to SQLite.")
# Close the connection
conn.close()

In [ ]:
# Stop the Spark session at end of script
spark.stop()